# Exploring Sherlock I/O Performance Inside an AI Dataloader

In many HPC workflows for training AI models, training and data processing pipelines are limited not only by compute, but also by I/O performance.

Large datasets are often stored on nearby filesystems such as `/scratch/`. These systems provide high capacity and are accessible from all compute nodes in the cluster. Accessing these files requires transferring data from `/scratch` to the compute node over the network. The network interface on each compute node has a finite bandwidth, and multiple jobs or processes running on the same node must share that connection. As a result, concurrent data access can become limited by the available network bandwidth rather than the raw performance of the filesystem itself, especially if many users are moving many files on and off at the same time.

Many clusters therefore provide node-local scratch storage. Sherlock provides this as local SSD storage which is available while a compute node is active at the `$L_SCRATCH` path. Because `$L_SCRATCH` is local, it can sometimes provide higher read performance, especially when repeatedly accessing many small files during machine learning training loops.

However, this benefit depends on the workload and cluster configuration. In particular:

- `$L_SCRATCH` is **most effective when running on a full compute node exclusively**
- Performance can degrade if **many users share the same node**
- **Data must first be copied or staged** from `/scratch/` to `$L_SCRATCH` adding up front time that is only mitigated when the same files are accessed more than one time

### Learning Goals:

In this notebook, we will simulate a common machine learning workflow and measure the difference between:
1. Streaming images directly from `$SCRATCH`
2. Copying the dataset to `$L_SCRATCH` and loading from there

We'll see how these changes scale when increasing batch sizes. 

## Computing Environment

Our computing environment uses many custom Python packages that have been built for working with geospatial imagery. To manage these packages on Sherlock, we're using a custom kernel that we created earlier in the session from a `conda` environment inside a container. To learn how to create your own custom environments like this, check out the [package management tutorial](https://github.com/stanford-sdss/package-management) from the [SDSS Center for Computation](https://sdss-compute.stanford.edu/).

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import Subset

import numpy as np
import rasterio

import time
import os
import shutil
from pathlib import Path

from concurrent.futures import ThreadPoolExecutor

Satellite imagery datasets often contain images of different spatial dimensions. For some models, you'll want to make sure you've preprocessed your imagery into slices of the same size. Here though, we're just demonstrating data loading without training a model, so it's okay if we load images of different sizes.

PyTorch’s default batching behavior expects tensors in a batch to have the same shape. To allow variable-sized images, we provide a custom collate utility function that simply returns a list of tensors rather than stacking them.

In [4]:
def collate_fn_list(batch):
    '''
    Allows pytorch to return files of different sizes
    '''
    return batch  # List of tensors with different sizes

## Defining a PyTorch Dataset for Landsat Images

Next we define a PyTorch `Dataset` class that loads Landsat imagery from disk. In this function we load `.tiff` files, a common file type for satellite imagery, into and RGB array in Python using the `rasterio` package. Then to prepare our data for AI training, we convert it into PyTorch tensors.

This structure mirrors typical machine learning workflows where a dataset object handles the on-demand streaming of images during training. This is necessary because the onboard memory of GPUs is typically not large enough to hold a full dataset in memory. As a result, you can choose where these images are streamed from. Later we'll test streaming them both from `/scratch/` and copying them first into `$L_SCRATCH` and streaming from there.

In [5]:
class LandsatDataset(Dataset):
    '''
    pytorch dataloader: https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html
    '''
    def __init__(self, image_dir):
        self.image_dir = Path(image_dir)
        self.image_paths = sorted(list(self.image_dir.glob('*.tif')) + 
                                  list(self.image_dir.glob('*.tiff')))
        print(f"Found {len(self.image_paths)} images in {image_dir}")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        with rasterio.open(img_path) as src:
            img = src.read()  # Read all bands
        img = torch.from_numpy(img.astype(np.float32))
        return img

## Timing our Performance

To measure I/O performance, we create a function that times how long it takes to iterate through an entire dataset in batches using a PyTorch DataLoader. This simulates a typical machine learning training loop where batches of data are continuously streamed from disk in a set batch size.

In [9]:
def time_dataloader(dataset, batch_size, num_epochs=1):
    '''
    Time how long it takes to iterate through the entire dataset
    '''
    dataloader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=True,
        num_workers=4, 
        pin_memory=False,
        collate_fn=collate_fn_list
    )
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        for batch_idx, batch in enumerate(dataloader):
            # Just load the data, don't process
            pass
    
    end_time = time.time()
    total_time = end_time - start_time
    
    num_batches = len(dataloader)
    time_per_batch = total_time / (num_batches * num_epochs)
    
    return {
        'total_time': total_time,
        'time_per_batch': time_per_batch,
        'num_batches': num_batches
    }

## A Storage Performance Experiment

We now define a function that performs the full experiment.

For a given batch size, we measure two scenarios:

**Scenario 1: Direct Streaming from `/scratch/`**

The dataloader pulls images into memory from Sherlock's `/scratch/` storage.

**Scenario 2: Copy to `$L_SCRATCH` First**

1. The dataset is copied from `/scratch/` to `$L_SCRATCH`.
2. The dataloader pulls images into memory from the local disk.

In [14]:
def run_experiment(batch_size, max_images=128):
    '''
    Run timing experiments for a given batch size
    '''
    print(f"\n{'='*60}")
    print(f"BATCH SIZE: {batch_size}")
    print(f"{'='*60}\n")
    
    scratch_path = "/scratch/groups/jfreshwa/workshops/io-for-large-datasets/landsat_images"
    l_scratch_path = f"{os.environ['L_SCRATCH']}/landsat_images"
    
    # Scenario 1: Load in directly from /scratch/
    print(f"Scenario 1: Loading directly from /scratch/")
    dataset1 = LandsatDataset(scratch_path)
    dataset1_subset = Subset(dataset1, range(max_images))
    results1 = time_dataloader(dataset1_subset, batch_size)
    print(f"  Total time: {results1['total_time']:.2f}s")
    print(f"  Time per batch: {results1['time_per_batch']:.3f}s")
    print(f"  Batches: {results1['num_batches']}")
    
    # Scenario 2: Copy to $L_SCRATCH first, then load
    print(f"\nScenario 2: Copy {max_images} images to $L_SCRATCH, then load")

    # Check if data already exists in $L_SCRATCH
    if os.path.exists(l_scratch_path):
        copy_time = None  # No copy needed
    else:
        copy_time = "needs_copy"
    
    # Only copy if needed
    if copy_time == "needs_copy":
        copy_start = time.time()
        
        # Create destination directory
        os.makedirs(l_scratch_path, exist_ok=True)
        
        # Get first max_images files
        scratch_dir = Path(scratch_path)
        image_files = sorted(list(scratch_dir.glob('*.tif')) + list(scratch_dir.glob('*.tiff')))[:max_images]
        
        num_workers = int(os.environ.get('SLURM_CPUS_PER_TASK', os.cpu_count()))
        print(f"  Copying {len(image_files)} files using {num_workers} workers...")
        
        # Parallel copy using threads
        def copy_one_file(img_file):
            shutil.copy2(img_file, l_scratch_path)
        
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            executor.map(copy_one_file, image_files)
        
        copy_time = time.time() - copy_start
        print(f"  Copy time: {copy_time:.2f}s")
    
    dataset2 = LandsatDataset(l_scratch_path)
    dataset2_subset = Subset(dataset2, range(max_images))
    results2 = time_dataloader(dataset2_subset, batch_size)
    print(f"  DataLoader time: {results2['total_time']:.2f}s")
    if copy_time is not None:
        print(f"  Total time (copy + load): {copy_time + results2['total_time']:.2f}s")
    
    print(f"  Time per batch: {results2['time_per_batch']:.3f}s")
    print(f"  Time per batch: {results2['time_per_batch']:.3f}s")

    # Print out timing summary
    print(f"\n{'='*60}")
    print(f"SUMMARY (Batch Size {batch_size}):")
    print(f"{'='*60}")
    print(f"Direct from /scratch/:           {results1['total_time']:.2f}s ({results1['time_per_batch']:.3f}s/batch)")
    if copy_time is not None:
        print(f"Copy + Load from $L_SCRATCH:    {copy_time + results2['total_time']:.2f}s")
        print(f"  - Copy time:                  {copy_time:.2f}s")
        print(f"  - Load time:                  {results2['total_time']:.2f}s ({results2['time_per_batch']:.3f}s/batch)")
    else:
        print(f"Load from $L_SCRATCH (cached):  {results2['total_time']:.2f}s ({results2['time_per_batch']:.3f}s/batch)")
    
    print(f"Load speedup ($L_SCRATCH):      {results1['total_time']/results2['total_time']:.2f}x")
  
    return {
        'batch_size': batch_size,
        'scratch': results1,
        'copy_and_load': {'copy_time': copy_time, **results2}
    }

## Testing Timing Across a Variety of Batch Sizes

Finally, we run the experiment for several batch sizes.

Batch size can influence data loading behavior because each batch requires multiple images to be read from disk. In practice, the performance impact of batch size depends on whether the workload is limited by storage bandwidth, CPU-based image decoding, or memory operations.

In [15]:
results_1 = run_experiment(batch_size=1, max_images=128)
results_32 = run_experiment(batch_size=32, max_images=128)
results_64 = run_experiment(batch_size=64, max_images=128)


BATCH SIZE: 1

Scenario 1: Loading directly from /scratch/
Found 466 images in /scratch/groups/jfreshwa/workshops/io-for-large-datasets/landsat_images
  Total time: 62.84s
  Time per batch: 0.491s
  Batches: 128

Scenario 2: Copy 128 images to $L_SCRATCH, then load
  Copying 128 files using 7 workers...
  Copy time: 1.49s
Found 128 images in /lscratch/ellianna/landsat_images
  DataLoader time: 57.47s
  Total time (copy + load): 58.96s
  Time per batch: 0.449s
  Time per batch: 0.449s

SUMMARY (Batch Size 1):
Direct from /scratch/:           62.84s (0.491s/batch)
Copy + Load from $L_SCRATCH:    58.96s
  - Copy time:                  1.49s
  - Load time:                  57.47s (0.449s/batch)
Load speedup ($L_SCRATCH):      1.09x

BATCH SIZE: 32

Scenario 1: Loading directly from /scratch/
Found 466 images in /scratch/groups/jfreshwa/workshops/io-for-large-datasets/landsat_images
  Total time: 72.78s
  Time per batch: 18.196s
  Batches: 4

Scenario 2: Copy 128 images to $L_SCRATCH, then

## Understanding Data-Heavy Pipeline Bottlenecks in AI Training

When analyzing data loading performance, there are typically three dominant bottlenecks:

1. **Storage I/O** – the raw speed at which data can be read from disk or over the network.
2. **File metadata / inode operations** – opening a large number of small files can be limited by the filesystem’s ability to manage directory entries and metadata.
3. **Image decoding / preprocessing** – reading formats like GeoTIFF or JPEG often requires CPU time to decompress and convert data into tensors.

In this notebook, we only load 128 images for the sake of time, so the overall loading time is mostly dominated by image decoding in CPU space rather than I/O bottlenecks that occur when loading in data from storage. That is why the difference between `/scratch/` and `$L_SCRATCH` is modest. However, as datasets scale to 10s or 100s of thousands of files, storage and metadata operations become much more significant. Reading 100K+ images directly from storage like `/scratch/` can saturate the network bandwidth and slow down training substantially.

Staging data to `$L_SCRATCH` in those cases allows the compute node to read files from a fast local disk, eliminating network and metadata bottlenecks for repeated access, especially when you are using the entire node for compute already. Over large training runs, the initial copy cost is quickly amortized, leading to faster, more stable, and predictable data loading performance, which is critical for high-scale machine learning and scientific computing pipelines.